# S6E9 complete V100 CT outer caches
Private verification only. No competition submission.

In [ ]:
from pathlib import Path
import subprocess,sys
work=Path("/kaggle/working")
(work/'cache_runner.py').write_text('#!/usr/bin/env python3\n"""CT outer caches. fit_outer accepts y_train only; never scores holdouts."""\nfrom __future__ import annotations\nimport os\nfor key in (\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\',\'NUMEXPR_NUM_THREADS\'):\n    os.environ[key]=\'4\'\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport sys\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold\nfrom ct_features import fit_features, transform_features, binary_target\n\nROOT=Path(__file__).resolve().parent\n\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda:f.read(1024*1024),b\'\'): h.update(chunk)\n    return h.hexdigest()\n\ndef array_sha(a):\n    return hashlib.sha256(np.asarray(a,dtype=\'<i8\').tobytes()).hexdigest()\n\ndef atomic_json(path,obj):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\')\n    with tmp.open(\'w\') as f: json.dump(obj,f,indent=2,allow_nan=False);f.write(\'\\n\')\n    os.replace(tmp,path)\n\ndef atomic_npz(path,**items):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    tmp=path.with_name(path.name+\'.tmp\')\n    with tmp.open(\'wb\') as f: np.savez_compressed(f,**items)\n    os.replace(tmp,path)\n\ndef probability(x,n):\n    x=np.asarray(x,dtype=np.float64)\n    assert x.shape==(n,) and np.isfinite(x).all() and x.min()>=0 and x.max()<=1\n    return x\n\ndef gpu_model(cfg):\n    import ctboost\n    return ctboost.CTBoostClassifier(iterations=cfg[\'iterations\'],task_type=\'GPU\',devices=\'0\',random_seed=cfg[\'model_seed\'],eval_metric=\'AUC\',verbose=False,**cfg[\'params\'])\n\ndef predict_fit(x,y,queries,original,cfg,factory):\n    assert \'Will_Buy_EV\' not in x\n    assert all(\'Will_Buy_EV\' not in q for q in queries)\n    features,state=fit_features(x,y,original,seed=cfg[\'model_seed\'])\n    model=factory(cfg).fit(features,y)\n    outputs=[probability(model.predict_proba(transform_features(q,state))[:,1],len(q)) for q in queries]\n    if cfg.get(\'require_gpu\',True):\n        assert model.get_booster()._handle.export_state()[\'task_type\']==\'GPU\'\n    del features,state,model\n    gc.collect()\n    return outputs\n\ndef fit_outer(x_train,y_train,x_valid,original,train_idx,valid_idx,cfg,out,factory=gpu_model):\n    # U labels are deliberately absent from this function and all callees.\n    assert len(x_train)==len(y_train)==len(train_idx) and len(x_valid)==len(valid_idx)\n    assert not np.intersect1d(train_idx,valid_idx).size\n    out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    identity={\'train_idx_sha256\':array_sha(train_idx),\'valid_idx_sha256\':array_sha(valid_idx),\n              \'train_id_sha256\':array_sha(x_train.id),\'valid_id_sha256\':array_sha(x_valid.id),\n              \'config_sha256\':cfg[\'_config_sha256\']}\n    # Historical CT OOF was stored as float32; retain its exact meta-input precision.\n    oof=np.full(len(x_train),np.nan,dtype=np.float32);fold_ids=np.full(len(x_train),-1,dtype=np.int8)\n    mean=np.zeros(len(x_valid),dtype=np.float64);rows=[]\n    folds=StratifiedKFold(cfg[\'inner_folds\'],shuffle=True,random_state=cfg[\'inner_seed\'])\n    for atom,(fit,hold) in enumerate(folds.split(np.zeros(len(y_train)),y_train),1):\n        cp=out/f\'atom_{atom:02d}.npz\';manifest=out/f\'atom_{atom:02d}.json\'\n        if cp.exists() or manifest.exists():\n            assert cp.exists() and manifest.exists(),\'incomplete checkpoint pair\'\n            record=json.loads(manifest.read_text());assert record[\'identity\']==identity and record[\'sha256\']==sha(cp)\n            with np.load(cp,allow_pickle=False) as z:\n                assert np.array_equal(z[\'fit_idx\'],train_idx[fit]) and np.array_equal(z[\'hold_idx\'],train_idx[hold])\n                assert np.array_equal(z[\'valid_idx\'],valid_idx)\n                a=probability(z[\'oof_proba\'],len(hold));b=probability(z[\'valid_proba\'],len(x_valid))\n        else:\n            tic=time.monotonic()\n            a,b=predict_fit(x_train.iloc[fit],y_train[fit],[x_train.iloc[hold],x_valid],original,cfg,factory)\n            atomic_npz(cp,fit_idx=train_idx[fit],hold_idx=train_idx[hold],valid_idx=valid_idx,oof_proba=a,valid_proba=b)\n            record={\'atom\':atom,\'seconds\':time.monotonic()-tic,\'identity\':identity,\'sha256\':sha(cp)}\n            atomic_json(manifest,record)\n        oof[hold]=a;fold_ids[hold]=atom-1;mean+=b/cfg[\'inner_folds\'];rows.append(record)\n        print(json.dumps({\'status\':\'CT_ATOM_COMPLETE\',\'outer\':out.name,**record}),flush=True)\n    cp=out/\'fullfit.npz\';manifest=out/\'fullfit.json\'\n    if cp.exists() or manifest.exists():\n        assert cp.exists() and manifest.exists(),\'incomplete fullfit checkpoint\'\n        record=json.loads(manifest.read_text());assert record[\'identity\']==identity and record[\'sha256\']==sha(cp)\n        with np.load(cp,allow_pickle=False) as z:\n            assert np.array_equal(z[\'valid_idx\'],valid_idx)\n            full=probability(z[\'valid_proba\'],len(x_valid))\n    else:\n        tic=time.monotonic()\n        full,=predict_fit(x_train,y_train,[x_valid],original,cfg,factory)\n        atomic_npz(cp,valid_idx=valid_idx,valid_proba=full)\n        record={\'seconds\':time.monotonic()-tic,\'identity\':identity,\'sha256\':sha(cp)}\n        atomic_json(manifest,record)\n    probability(oof,len(x_train));probability(mean,len(x_valid));assert (fold_ids>=0).all()\n    cache=out/\'cache.npz\'\n    atomic_npz(cache,train_idx=train_idx,valid_idx=valid_idx,train_id=x_train.id.to_numpy(),valid_id=x_valid.id.to_numpy(),\n               oof_proba=oof,valid_proba_fullfit=full,valid_proba_foldmean=mean,atom_fold=fold_ids)\n    summary={\'status\':\'CT_CACHE_COMPLETE_UNSCORED\',\'identity\':identity,\'cache_sha256\':sha(cache),\'atoms\':rows,\n             \'fullfit\':record,\'atom_fold_sha256\':array_sha(fold_ids),\'validation_labels_used\':False,\n             \'allowed_for_submission\':False,\'source_sha256\':cfg[\'source_sha256\']}\n    atomic_json(out/\'cache_manifest.json\',summary)\n    return summary\n\ndef run(input_root,output):\n    import ctboost\n    import sklearn\n    cfg_path=ROOT/\'frozen_config.json\';cfg=json.loads(cfg_path.read_text())\n    cfg[\'_config_sha256\']=sha(cfg_path)\n    for name,digest in cfg[\'code_sha256\'].items(): assert sha(ROOT/name)==digest,(\'code drift\',name)\n    assert ctboost.__version__==\'0.1.58\' and ctboost.build_info()[\'cuda_enabled\']\n    expected=cfg[\'remote_versions\'];actual={\'numpy\':np.__version__,\'pandas\':pd.__version__,\'sklearn\':sklearn.__version__}\n    assert actual==expected,(\'remote runtime drift\',actual,expected)\n    base=Path(input_root)\n    trains=list(base.glob(\'**/playground-series-s6e9/train.csv\'))\n    originals=list(base.glob(\'**/EV_Adoption_and_Range_Anxiety_Dataset.csv\'))\n    assert len(trains)==1 and len(originals)==1,(trains,originals)\n    assert sha(trains[0])==cfg[\'source_sha256\'][\'train.csv\']\n    assert sha(originals[0])==cfg[\'source_sha256\'][\'original.csv\']\n    frame=pd.read_csv(trains[0]);y=binary_target(frame.Will_Buy_EV);x=frame.drop(columns=\'Will_Buy_EV\')\n    original=pd.read_csv(originals[0]);del frame\n    outer_results=[]\n    for outer,(train,valid) in enumerate(StratifiedKFold(cfg[\'outer_folds\'],shuffle=True,random_state=cfg[\'outer_seed\']).split(np.zeros(len(y)),y),1):\n        outer_results.append(fit_outer(x.iloc[train],y[train],x.iloc[valid],original,train,valid,cfg,Path(output)/f\'outer_{outer:02d}\'))\n    atomic_json(Path(output)/\'GPU_COMPLETE.json\',{\'status\':\'CT_CACHE_COMPLETE_UNSCORED\',\'config_sha256\':cfg[\'_config_sha256\'],\n        \'outer_results\':outer_results,\'versions\':actual,\'complete_v100_score\':None,\'submission_created\':False})\n\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser();ap.add_argument(\'--input-root\',default=\'/kaggle/input\');ap.add_argument(\'--output\',default=\'/kaggle/working/ct_cache\')\n    args=ap.parse_args()\n    run(args.input_root,args.output)\n')
(work/'ct_features.py').write_text('"""Verbatim feature functions from frozen historical CT notebook."""\nimport hashlib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.preprocessing import TargetEncoder\n\ndef binary_target(values):\n    values = pd.Series(np.asarray(values).reshape(-1))\n    if not pd.api.types.is_numeric_dtype(values.dtype):\n        values = values.map({"Yes": 1, "No": 0, "1": 1, "0": 0})\n    if values.isna().any() or not set(values.unique()).issubset({0, 1}):\n        raise ValueError("The target must contain only Yes/No or 1/0.")\n    return values.to_numpy(dtype=np.int8)\n\ndef numeric_values(values):\n    return pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)\n\ndef category_keys(values):\n    # A prefix keeps a missing value distinct from an ordinary category name.\n    return ("V:" + values.astype("string")).fillna("M:").astype(object)\n\ndef base_features(data, state):\n    missing = set(state["input_columns"]).difference(data.columns)\n    if missing:\n        raise ValueError(f"Missing input columns: {sorted(missing)}")\n    columns = {}\n    for name in state["input_columns"]:\n        if name in state["category_maps"]:\n            columns[name] = category_keys(data[name]).map(state["category_maps"][name]).fillna(-1)\n        else:\n            columns[name] = numeric_values(data[name])\n    for name in state["input_columns"]:\n        if name in state["category_maps"]:\n            continue\n        values = numeric_values(data[name]).fillna(0).to_numpy(dtype=np.float64)\n        for digit in range(-4, 4):\n            # Keep floor division: changing this arithmetic changes the recipe.\n            columns[f"{name}_digit{digit}"] = values // (10.0 ** digit) % 10\n    income = pd.to_numeric(data["Annual_Income_USD"], errors="coerce")\n    concern = pd.to_numeric(data["Environmental_Concern_Level"], errors="coerce")\n    columns["is_30k_spike"] = income == 30000.0\n    columns["is_millionaire_cliff"] = income >= 170537.0\n    columns["is_dead_zone"] = income.between(38000.0, 42000.0)\n    columns["is_env_hater"] = concern == 1\n    return pd.DataFrame(columns, index=data.index, dtype=np.float32)\n\ndef unique_columns(frame):\n    """Remove constants and exact duplicates, using training rows only."""\n    keep, seen = [], set()\n    for name in frame:\n        if frame[name].nunique(dropna=False) <= 1:\n            continue\n        values = frame[name].to_numpy(dtype=np.float32, copy=True)\n        values[values == 0] = 0\n        values[np.isnan(values)] = np.nan\n        digest = hashlib.blake2b(values.tobytes(), digest_size=16).digest()\n        if digest not in seen:\n            keep.append(name)\n            seen.add(digest)\n    return keep\n\ndef smooth_keys(data):\n    income = numeric_values(data["Annual_Income_USD"])\n    commute = numeric_values(data["Daily_Commute_km"])\n    values = {\n        "income_exact_integer": income,\n        "income100_floor": np.floor(income / 100),\n        "income1000_floor": np.floor(income / 1000),\n        "commute_integer": np.floor(commute),\n    }\n    return pd.DataFrame({\n        name: category_keys(numeric_values(value).round().astype("Int64"))\n        for name, value in values.items()\n    }, index=data.index)\n\ndef fit_maps(data, base, original, state):\n    original = original.assign(Will_Buy_EV=binary_target(original["Will_Buy_EV"]))\n    state["original_mean"] = float(original["Will_Buy_EV"].mean())\n    state["original_maps"] = {\n        name: original.groupby(name, observed=True, dropna=False)["Will_Buy_EV"].mean().to_dict()\n        for name in state["input_columns"] if name in original\n    }\n    state["core_columns"] = unique_columns(base)\n    state["encoding_columns"] = [name for name in state["core_columns"] if not name.startswith("is_")]\n    state["frequency_maps"] = {\n        name: base[name].value_counts(normalize=True, dropna=False).to_dict()\n        for name in state["encoding_columns"]\n    }\n\ndef encoded_features(data, base, state, y=None):\n    columns = {\n        name: base[name].to_numpy(dtype=np.float32, copy=False)\n        for name in state["core_columns"] if name not in state["category_maps"]\n    }\n    for name, mapping in state["original_maps"].items():\n        columns[f"{name}_org_mean"] = data[name].map(mapping).fillna(state["original_mean"])\n    for name in state["encoding_columns"]:\n        columns[f"{name}_fe"] = base[name].map(state["frequency_maps"][name]).fillna(0)\n    codes = base[state["encoding_columns"]]\n    for label in ("auto", "10"):\n        encoder = state["encoders"][label]\n        values = encoder.fit_transform(codes, y) if y is not None else encoder.transform(codes)\n        for index, name in enumerate(codes.columns):\n            columns[f"{name}_TE_{label}"] = values[:, index]\n    keys = smooth_keys(data)\n    encoder = state["encoders"]["100"]\n    values = encoder.fit_transform(keys, y) if y is not None else encoder.transform(keys)\n    for index, name in enumerate(keys.columns):\n        columns[f"enc_smooth_{name}_TE_100"] = values[:, index]\n    return pd.DataFrame(columns, index=data.index, dtype=np.float32)\n\ndef transform_features(data, state):\n    """Apply training mappings to validation or test rows without fitting."""\n    features = encoded_features(data, base_features(data, state), state)\n    return features[state["feature_names"]]\n\ndef fit_features(data, y, original_frame, seed=20260904):\n    """Return cross-fitted training features and a portable dictionary of state."""\n    if not isinstance(data, pd.DataFrame):\n        raise TypeError("Training data must be a pandas DataFrame.")\n    y = binary_target(y)\n    if len(data) != len(y):\n        raise ValueError("Training data and target have different lengths.")\n    folds = min(5, int(np.bincount(y, minlength=2).min()))\n    if folds < 2:\n        raise ValueError("Target encoding needs at least two examples of each class.")\n    excluded = {"id", "Will_Buy_EV", "is_train", "Number_of_Cars_Owned"}\n    names = [name for name in data if name not in excluded]\n    categories = [name for name in names if not pd.api.types.is_numeric_dtype(data[name].dtype)]\n    state = {"input_columns": names, "category_maps": {}, "encoders": {}, "seed": seed}\n    for name in categories:\n        values = sorted(category_keys(data[name]).unique())\n        state["category_maps"][name] = {value: index for index, value in enumerate(values)}\n    base = base_features(data, state)\n    fit_maps(data, base, original_frame, state)\n    for label, smooth in (("auto", "auto"), ("10", 10.0), ("100", 100.0)):\n        state["encoders"][label] = TargetEncoder(\n            target_type="binary", smooth=smooth, cv=5 if label == "100" else folds,\n            shuffle=True, random_state=seed,\n        )\n    features = encoded_features(data, base, state, y)\n    state["feature_names"] = list(features.columns)\n    return features, state\n')
(work/'supervisor.py').write_text('"""One process tree, hard wall/RSS guards; preserve remote partial output."""\nimport fcntl\nimport json\nimport os\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\nimport psutil\n\nROOT=Path(__file__).resolve().parent\nLIMIT=7200\nMEMORY=24*1024**3\nstarted=time.monotonic()\npeak=0\nlock=(ROOT/\'gpu_run.lock\').open(\'a\')\nfcntl.flock(lock,fcntl.LOCK_EX|fcntl.LOCK_NB)\nmarker=ROOT/\'GPU_RUN_STARTED.json\'\nassert not marker.exists(),\'do not start a duplicate run in the same workspace\'\nmarker.write_text(json.dumps({\'pid\':os.getpid(),\'started_unix\':time.time(),\'budget_seconds\':LIMIT,\'memory_bytes\':MEMORY}))\n\ndef kill_tree(process):\n    if process.poll() is None:\n        os.killpg(process.pid,signal.SIGTERM)\n        try:process.wait(timeout=3)\n        except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()\n\ndef guarded(script):\n    global peak\n    process=subprocess.Popen([sys.executable,\'-u\',str(ROOT/script)],cwd=ROOT,start_new_session=True)\n    try:\n        while process.poll() is None:\n            root=psutil.Process(process.pid)\n            descendants=[root,*root.children(recursive=True)]\n            rss=0\n            for child in descendants:\n                try:rss+=child.memory_info().rss\n                except psutil.NoSuchProcess:pass\n            peak=max(peak,rss)\n            if time.monotonic()-started>=LIMIT:raise TimeoutError(\'7200 second total GPU budget exceeded\')\n            if rss>MEMORY:raise MemoryError(\'24 GiB process-tree memory budget exceeded\')\n            time.sleep(.5)\n        if process.returncode:raise RuntimeError(f\'{script} failed with status {process.returncode}\')\n    finally:kill_tree(process)\n\ntry:\n    guarded(\'bootstrap.py\')\n    guarded(\'cache_runner.py\')\n    status={\'status\':\'GPU_CACHE_RUN_COMPLETE\',\'seconds\':time.monotonic()-started,\'peak_process_tree_rss\':peak}\nexcept BaseException as exc:\n    status={\'status\':\'GPU_CACHE_RUN_FAILED\',\'type\':type(exc).__name__,\'error\':str(exc),\'seconds\':time.monotonic()-started,\'peak_process_tree_rss\':peak}\n    (ROOT/\'GPU_RUN_RESULT.json\').write_text(json.dumps(status,indent=2))\n    raise\nelse:(ROOT/\'GPU_RUN_RESULT.json\').write_text(json.dumps(status,indent=2))\nfinally:fcntl.flock(lock,fcntl.LOCK_UN);lock.close()\n')
(work/'bootstrap.py').write_text('import hashlib\nimport importlib.util\nimport json\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nfrom pathlib import Path\n\ndef pip_install(*packages):\n    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)\n    importlib.invalidate_caches()\n\npackages = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn>=1.3",\n            "joblib": "joblib", "matplotlib": "matplotlib"}\nmissing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]\nif missing:\n    pip_install(*missing)\n\nencoder_check = subprocess.run(\n    [sys.executable, "-c", "from sklearn.preprocessing import TargetEncoder"], capture_output=True)\nif encoder_check.returncode:\n    if any(name in sys.modules for name in ("sklearn", "numpy", "scipy")):\n        raise RuntimeError("This session needs scikit-learn >=1.3. Start a fresh session before updating it.")\n    pip_install("scikit-learn>=1.3")\n\nprobe = subprocess.run(\n    [sys.executable, "-c", "import ctboost,json; print(json.dumps(ctboost.build_info()))"],\n    capture_output=True, text=True)\ntry:\n    installed = json.loads(probe.stdout.strip().splitlines()[-1])\nexcept (ValueError, IndexError):\n    installed = {}\n\nif any(installed.get(key) != "0.1.58" for key in ("version", "package_version")) or not installed.get("cuda_enabled"):\n    if "ctboost" in sys.modules:\n        raise RuntimeError("Restart the notebook session, then Run All to load the CTBoost GPU build.")\n    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"\n    wheel_hashes = {\n        "cp310": "c0a67a5e2d91d74c6f72566f79b3d4da4d8145c0a94b004c2603b0f41c94de79",\n        "cp311": "3ee9a28777d1a3b7afdaadaeb7a46204164cdaffd8493bfcf9990a56bc815a61",\n        "cp312": "ff868712ac6b93f9038c646aedfe9fba0b9a9e7b95297ae0aa211157ee014a8b",\n        "cp313": "36edbf6674a2cc6ce5fdf21caba40fa589e672712d20381dce88b70defd8faeb",\n        "cp314": "d334bba5296731800ff24f782c939e8631a0afecc585a95277987357e443136c",\n    }\n    if tag not in wheel_hashes or not sys.platform.startswith("linux"):\n        raise RuntimeError("Run this notebook in a Kaggle Linux GPU session with Python 3.10–3.14.")\n    wheel = Path("/tmp") / f"ctboost-0.1.58-{tag}-{tag}-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"\n    url = "https://github.com/captnmarkus/ctboost/releases/download/v0.1.58/" + wheel.name\n    for attempt in range(3):\n        try:\n            if not wheel.exists() or hashlib.sha256(wheel.read_bytes()).hexdigest() != wheel_hashes[tag]:\n                request = urllib.request.Request(url, headers={"User-Agent": "CTBoost-notebook"})\n                with urllib.request.urlopen(request, timeout=120) as response:\n                    wheel.write_bytes(response.read())\n            if hashlib.sha256(wheel.read_bytes()).hexdigest() != wheel_hashes[tag]:\n                raise RuntimeError("Incomplete wheel download. Check Internet access and rerun setup.")\n            break\n        except Exception:\n            if attempt == 2:\n                raise\n            time.sleep(2)\n    pip_install("--force-reinstall", "--no-deps", str(wheel))\n\n\nimport gc\nimport joblib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport sklearn\nimport ctboost\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.preprocessing import TargetEncoder\nfrom IPython.display import display\n\nbuild = ctboost.build_info()\nif ctboost.__version__ != "0.1.58" or any(build.get(key) != "0.1.58" for key in ("version", "package_version")) or not build.get("cuda_enabled"):\n    raise RuntimeError("Restart the session and Run All to load CTBoost 0.1.58 GPU.")\ngpu_data = np.random.default_rng(42).normal(size=(128, 4)).astype(np.float32)\ngpu_check = ctboost.CTBoostClassifier(iterations=3, max_depth=2, task_type="GPU", verbose=False)\ngpu_check.fit(gpu_data, (gpu_data[:, 0] > 0).astype(np.int32))\nassert gpu_check.get_booster()._handle.export_state()["task_type"] == "GPU"\ndel gpu_check, gpu_data\nplt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})\nprint(f"CTBoost {ctboost.__version__} GPU is ready | scikit-learn {sklearn.__version__}")\n\n\n')
(work/'frozen_config.json').write_text('{\n  "experiment_id": "E2E_V100_CT_CACHE_20260905",\n  "outer_folds": 5,\n  "outer_seed": 42437,\n  "inner_folds": 5,\n  "inner_seed": 42,\n  "iterations": 1426,\n  "model_seed": 20260904,\n  "params": {\n    "learning_rate": 0.039,\n    "max_depth": 4,\n    "max_leaves": 16,\n    "grow_policy": "LeafWise",\n    "alpha": 0.5,\n    "lambda_l2": 8.0,\n    "min_data_in_leaf": 100,\n    "min_child_weight": 0.1,\n    "subsample": 0.85,\n    "bootstrap_type": "Bernoulli",\n    "colsample_bytree": 0.3,\n    "feature_test": "quadratic",\n    "feature_test_adjustment": "none",\n    "feature_test_bins": 8,\n    "max_bins": 1024,\n    "leaf_estimation_iterations": 3\n  },\n  "require_gpu": true,\n  "remote_versions": {\n    "numpy": "2.0.2",\n    "pandas": "2.3.3",\n    "sklearn": "1.6.1"\n  },\n  "source_sha256": {\n    "train.csv": "eae9eaa4e6378df405e755f853771d7e26d212bd93258349fc797b771021946a",\n    "original.csv": "c271a380df51b18177d0a039d54525ca7c3d71500701dc7496714a091df16fae",\n    "historical_ct_notebook": "a78ce26d2f8f6c23bf15edb317f457d3e568823a7a92769f397b270bfb608209"\n  },\n  "code_sha256": {\n    "cache_runner.py": "8ed44cef69472c575f6981dca53e791833b1acbcf9a374b01cd7b79c8c40521b",\n    "ct_features.py": "7cae5668f9c86938dc63635cfcfc7d4973059e48b754241cf8f88a5d220e2b14",\n    "supervisor.py": "e2060f3c148ea57cfdccb0a9c94f61bd8732747f3cae6f7bf4c5706a80ea0662",\n    "bootstrap.py": "8e9ab142c39982cc81e3656e0e5cafe45246e888543cdac405a2e2fc4d5590f0"\n  },\n  "budget_seconds": 7200,\n  "memory_bytes": 25769803776,\n  "threads": 4,\n  "submission_budget": 0,\n  "outer_scores_not_computed": true,\n  "preregistration_sha256": "d5677c282d891c89abd4ffee0ce9aa18ec92304c3143939051f1fe936787bf8b",\n  "synthetic_test_sha256": "78f2a594812da60d1b0acdcbb4e937e8d6138cf1907ae72515b4c221da8a26a4"\n}\n')
subprocess.run([sys.executable,"-u",str(work/"supervisor.py")],cwd=work,check=True)
